# CV Threshold Accuracy
Cross-validated accuracy within error thresholds.


In [1]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

MODEL_DIR = 'xgboost'
MODEL_NAME = 'XGBoost'

cwd = Path.cwd()
project_root = Path(".." ).resolve() if cwd.name == MODEL_DIR else cwd
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from qspr_config import (
    DATA_PATH,
    ECFP_RADIUS,
    ECFP_N_BITS,
    N_ESTIMATORS,
    RANDOM_SEED,
    CV_FOLDS,
    N_JOBS,
    FIG_DPI,
    FIGSIZE_SQUARE,
)
from qspr_common import (
    load_dataset,
    build_feature_matrix,
    apply_plot_style,
    resolve_output_dir,
)

apply_plot_style()
out_dir = resolve_output_dir(MODEL_DIR)


In [2]:
df = load_dataset(DATA_PATH)
df, X = build_feature_matrix(df, radius=ECFP_RADIUS, n_bits=ECFP_N_BITS)
y = df["Solubility"].to_numpy()


[22:52:39] WARNING: not removing hydrogen atom without neighbors
[22:52:39] WARNING: not removing hydrogen atom without neighbors
[22:52:39] WARNING: not removing hydrogen atom without neighbors
[22:52:40] WARNING: not removing hydrogen atom without neighbors
[22:52:40] WARNING: not removing hydrogen atom without neighbors
[22:52:40] WARNING: not removing hydrogen atom without neighbors
[22:52:40] WARNING: not removing hydrogen atom without neighbors
[22:52:40] WARNING: not removing hydrogen atom without neighbors
[22:52:40] WARNING: not removing hydrogen atom without neighbors
[22:52:40] WARNING: not removing hydrogen atom without neighbors
[22:52:40] WARNING: not removing hydrogen atom without neighbors
[22:52:40] WARNING: not removing hydrogen atom without neighbors
[22:52:40] WARNING: not removing hydrogen atom without neighbors
[22:52:41] WARNING: not removing hydrogen atom without neighbors
[22:52:41] WARNING: not removing hydrogen atom without neighbors
[22:52:41] WARNING: not r

In [ ]:
from sklearn.model_selection import KFold
from xgboost import XGBRegressor

cv = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_SEED)
thresholds = [0.3, 0.5, 1.0]

fold_acc = {thr: [] for thr in thresholds}

for train_idx, test_idx in cv.split(X):
    model = XGBRegressor(
        n_estimators=N_ESTIMATORS,
        random_state=RANDOM_SEED,
        n_jobs=N_JOBS,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="reg:squarederror",
        verbosity=0,
    )
    model.fit(X[train_idx], y[train_idx])
    pred = model.predict(X[test_idx])
    err = np.abs(pred - y[test_idx])
    for thr in thresholds:
        fold_acc[thr].append((err <= thr).mean())

means = [np.mean(fold_acc[thr]) for thr in thresholds]
stds = [np.std(fold_acc[thr]) for thr in thresholds]

fig, ax = plt.subplots(figsize=FIGSIZE_SQUARE)
ax.bar(range(len(thresholds)), means, yerr=stds, capsize=4, color="#4C78A8")
ax.set_xticks(range(len(thresholds)))
ax.set_xticklabels([f"|err| <= {thr}" for thr in thresholds])
ax.set_ylabel("Accuracy")
ax.set_ylim(0, 1.0)
ax.set_title(f"{MODEL_NAME}: CV accuracy within threshold")
fig.tight_layout()

out_path = out_dir / "cv_threshold_accuracy.png"
fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight")
plt.show()
out_path
